# OND trigger test on v5 reanalysis

The adopted trigger (notebook 04) is calibrated on GloFAS v4 reanalysis
because the operational forecast the watch reads runs at v4 scale. This
notebook applies the SAME construction to v5 reanalysis: per-river top 3
seasons, lowered within the overall activation years (no new year can enter),
any-river union. It answers two questions:

- which years would the rule activate on if the record were v5, and
- how those years compare with the v4 activation years and the impact records.

Data: v5 Somali-box master (`glofas_discharge_v5.parquet`, 8 stations) +
`glofas_discharge_newbasins_ond_v5.parquet` (`extract_newbasins_v5.py`,
which also channel-checks each cell on the v5 mean field: all six sit at
0.95-1.00 of their window's max mean, so no re-snapping was needed).

This is a reference comparison, not a re-calibration: monitoring stays on v4
levels until GloFAS upgrades the operational system (the dashboard's
climatology check watches for that), at which point this notebook is the
starting point for re-fitting.

In [1]:
import ocha_stratus as stratus
import pandas as pd

STAGE = "dev"
PROJECT_PREFIX = "ds-aa-eth-flooding"
OND = [10, 11, 12]
YEARS = list(range(2003, 2026))
N_YEARS = len(YEARS)
TOP_N = 3

RIVER_OF = {
    "station_1": "Wabi Shebelle", "station_2": "Wabi Shebelle", "station_3": "Wabi Shebelle",
    "station_4": "Wabi Shebelle", "G1904": "Wabi Shebelle",
    "RP_4045_0535": "Genale Dawa", "RP_4205_0415": "Genale Dawa",
    "station_5": "Genale Dawa", "dawa_1": "Genale Dawa",
    "omo_1": "Omo", "bilate_1": "Bilate", "gamo_1": "Gamo lakes",
    "baro_1": "Baro", "akobo_1": "Akobo",
}
RIVERS = ["Wabi Shebelle", "Genale Dawa", "Omo", "Bilate", "Gamo lakes", "Baro", "Akobo"]

# the v4 adopted trigger's record, for comparison (notebook 04)
V4_UNION = {2008, 2011, 2013, 2017, 2019, 2023, 2024, 2025}
EMDAT_YEARS = {2006, 2007, 2008, 2011, 2015, 2019, 2023}
CERF_YEARS = {2006, 2018, 2020, 2023}
FS3_SOMALI = {2004, 2006, 2007, 2009, 2011, 2017, 2022, 2023}

## Load the v5 OND record, all 14 stations

In [2]:
som = stratus.load_parquet_from_blob(
    f"{PROJECT_PREFIX}/processed/glofas/glofas_discharge_v5.parquet", stage=STAGE
)[["station_id", "valid_time", "discharge"]]
nb_ = stratus.load_parquet_from_blob(
    f"{PROJECT_PREFIX}/processed/glofas/glofas_discharge_newbasins_ond_v5.parquet", stage=STAGE
)[["station_id", "valid_time", "discharge"]]
rean = pd.concat([som, nb_], ignore_index=True)
rean["valid_time"] = pd.to_datetime(rean["valid_time"]).dt.normalize()
rean["date"] = rean["valid_time"] - pd.Timedelta(days=1)
rean = rean.sort_values("valid_time").drop_duplicates(["station_id", "date"], keep="last")
ond = rean[rean["date"].dt.month.isin(OND) & rean["date"].dt.year.isin(YEARS)]
daily = ond.pivot_table(index="date", columns="station_id", values="discharge")
print(daily.shape, "|", sorted(daily.columns))

(2116, 14) | ['G1904', 'RP_4045_0535', 'RP_4205_0415', 'akobo_1', 'baro_1', 'bilate_1', 'dawa_1', 'gamo_1', 'omo_1', 'station_1', 'station_2', 'station_3', 'station_4', 'station_5']


## The same rule on v5: top 3 per river, lowered within the union

In [3]:
seas_max = daily.groupby(daily.index.year).max()
med_max = seas_max.median()
norm = seas_max / med_max
river_stat = pd.DataFrame({
    r: norm[[s for s, rr in RIVER_OF.items() if rr == r and s in norm.columns]].max(axis=1)
    for r in RIVERS
})

ratio_top3 = river_stat.apply(lambda col: col.sort_values(ascending=False).iloc[TOP_N - 1])
active_top3 = river_stat.ge(ratio_top3, axis=1)
union_top3 = set(active_top3.index[active_top3.any(axis=1)])
adj = {}
for r in RIVERS:
    ranked = river_stat[r].sort_values(ascending=False)
    keep = list(ranked.index[:TOP_N])
    for y in ranked.index[TOP_N:]:
        if y not in union_top3:
            break
        keep.append(y)
    adj[r] = ranked.loc[keep].min()
ratio_star = pd.Series(adj)[RIVERS]

active = river_stat.ge(ratio_star, axis=1)
union_years = sorted(active.index[active.any(axis=1)])
print(f"v5 union: {len(union_years)} activation years in {N_YEARS} | "
      f"RP {(N_YEARS + 1) / len(union_years):.2f}")
print("years:", union_years)
print("per-river activation seasons (v5):")
for r in RIVERS:
    yrs = active.index[active[r]].tolist()
    print(f"  {r:14s} ({(N_YEARS + 1) / len(yrs):.1f}-yr): {yrs}")

v5 union: 9 activation years in 23 | RP 2.67
years: [2008, 2014, 2017, 2019, 2020, 2021, 2023, 2024, 2025]
per-river activation seasons (v5):
  Wabi Shebelle  (8.0-yr): [2014, 2019, 2023]
  Genale Dawa    (4.0-yr): [2014, 2017, 2019, 2020, 2023, 2024]
  Omo            (6.0-yr): [2019, 2020, 2023, 2024]
  Bilate         (2.7-yr): [2008, 2014, 2017, 2019, 2020, 2021, 2023, 2024, 2025]
  Gamo lakes     (4.8-yr): [2019, 2020, 2023, 2024, 2025]
  Baro           (6.0-yr): [2008, 2019, 2024, 2025]
  Akobo          (8.0-yr): [2008, 2019, 2020]


## v5 vs v4 vs the impact records

In [4]:
v5_union = set(union_years)
rows = []
for tag, ys in [("v4 (adopted)", V4_UNION), ("v5 (this test)", v5_union)]:
    rows.append({
        "record": tag, "n_activations": len(ys),
        "overall_rp": round((N_YEARS + 1) / len(ys), 2),
        "emdat": f"{len(ys & EMDAT_YEARS)} of {len(EMDAT_YEARS)}",
        "cerf": f"{len(ys & CERF_YEARS)} of {len(CERF_YEARS)}",
        "floodscan_rp3_somali": f"{len(ys & FS3_SOMALI)} of {len(FS3_SOMALI)}",
        "years": ", ".join(str(y) for y in sorted(ys)),
    })
print("only in v5:", sorted(v5_union - V4_UNION))
print("only in v4:", sorted(V4_UNION - v5_union))
print("in both   :", sorted(V4_UNION & v5_union))
pd.DataFrame(rows)

only in v5: [2014, 2020, 2021]
only in v4: [2011, 2013]
in both   : [2008, 2017, 2019, 2023, 2024, 2025]


,record,n_activations,overall_rp,emdat,cerf,floodscan_rp3_somali,years
0,v4 (adopted),8,3.00,4 of 7,1 of 4,3 of 8,"2008, 2011, 2013, 2017, 2019, 2023, 2024, 2025"
1,v5 (this test),9,2.67,3 of 7,2 of 4,2 of 8,"2008, 2014, 2017, 2019, 2020, 2021, 2023, 2024..."


## What the v5 test shows

- The same construction on v5 gives 9 activation years (1-in-2.67): six shared
  with v4 (2008, 2017, 2019, 2023, 2024, 2025), plus 2014, 2020 and 2021;
  it loses v4's 2011 and 2013.
- Against the impact records, v5 does not score better overall: EM-DAT years
  caught drop from 4 to 3 (2011 is an EM-DAT and FloodScan RP3 year), FloodScan
  RP3 from 3 to 2, while CERF rises from 1 to 2 (v5 catches 2020, a CERF flood
  year with USD 8.0M allocated). 2014 and 2021 have no EM-DAT, CERF or
  FloodScan RP3 entry.
- 2006, the largest miss in the v4 backtest (EM-DAT 361,600 affected, 2 CERF
  allocations, FloodScan RP2+RP3), is still not caught on v5: the Shabelle's
  2006 season ranks 4th at 97 percent of its top-3 level.
- The union-constrained lowering behaves differently on v5: because the v5
  union is wider, the Bilate chains down to 9 kept seasons (1-in-2.7
  individually). Depth would need re-choosing, not just re-ranking, in any
  v5 re-calibration.
- Channel check: all six extracted cells sit at 0.95 to 1.00 of their
  window's maximum mean discharge on the v5 field, so the station registry
  carries over without re-snapping.